# Entrenamiento YOLO11: enfermedades de palta Hass

Se entrenan **dos modelos de detección**: uno para **hojas** y otro para **frutos**.
Los datasets se generan en local con `scripts/02_preparar_datasets.py` (ver `ESTRATEGIA_DATASET.md`).

**Antes de empezar**
1. Las carpetas `DatasetHojas/` y `DatasetFrutos/` deben estar en Google Drive, dentro de `CARPETA_DRIVE` (celda 2).
2. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4 o mejor)*.
3. *Entorno de ejecución → Ejecutar todas*. Entrena hojas, luego frutos, y muestra los resultados.

Todo se guarda en Drive, en `CARPETA_DRIVE/runs/`. Los modelos finales quedan en `runs/hojas/weights/best.pt` y `runs/frutos/weights/best.pt`.

In [ ]:
!nvidia-smi

## 1. Instalación

In [ ]:
%pip install "ultralytics<=8.3.40"
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## 2. Datasets desde Google Drive

In [ ]:
import os, re, shutil
from google.colab import drive

drive.mount('/content/drive')

CARPETA_DRIVE = '/content/drive/MyDrive/datasets/datasetEndermedadesPalta/PaltaHass'   # donde estan DatasetHojas/ y DatasetFrutos/
DATASETS = '/content/datasets'
os.makedirs(DATASETS, exist_ok=True)

for nombre in ('DatasetHojas', 'DatasetFrutos'):
    origen = f'{CARPETA_DRIVE}/{nombre}'
    if not os.path.exists(f'{origen}/data.yaml'):
        raise FileNotFoundError(f'No encuentro {origen}/data.yaml: revisa CARPETA_DRIVE')
    # Se copia al disco local de Colab: entrenar leyendo desde Drive es mucho mas lento.
    destino = f'{DATASETS}/{nombre}'
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    # 'path' del data.yaml apunta a la ruta de la PC local: se cambia por la de Colab.
    yaml = f'{destino}/data.yaml'
    texto = re.sub(r'^path:.*$', f'path: {destino}', open(yaml).read(), flags=re.M)
    open(yaml, 'w').write(texto)
    for split in ('train', 'valid', 'test'):
        print(f'{nombre}/{split}: {len(os.listdir(f"{destino}/{split}/images"))} imagenes')
    print(texto)


## 3. Entrenamiento

- `imgsz=1024`: resolución del dataset.
- `epochs=150` con `patience=30`: se detiene solo si deja de mejorar.
- `batch=-1`: ajusta el lote a la memoria de la GPU.
- Si `yolo11s` queda corto, prueba `yolo11m.pt`.

**Si Colab se desconecta:** vuelve a ejecutar las celdas 1 (instalación) y 2 (Drive), y luego la celda de entrenamiento que se cortó. Retoma desde la última época guardada en Drive. Si el entrenamiento ya había terminado, mostrará que no hay nada que retomar: eso es normal.

**Para entrenar un modelo desde cero** borra antes su carpeta `runs/hojas` o `runs/frutos` en Drive.

In [ ]:
%cd /content
import os
RUNS = f'{CARPETA_DRIVE}/runs'   # se guarda en Drive: si Colab se desconecta, no se pierde
last = f'{RUNS}/hojas/weights/last.pt'

if os.path.exists(last):
    print('Retomando entrenamiento desde', last)
    !yolo train resume model={last}
else:
    !yolo task=detect mode=train model=yolo11s.pt data={DATASETS}/DatasetHojas/data.yaml epochs=150 imgsz=1024 patience=30 batch=-1 plots=True project={RUNS} name=hojas exist_ok=True

In [ ]:
%cd /content
import os
RUNS = f'{CARPETA_DRIVE}/runs'   # se guarda en Drive: si Colab se desconecta, no se pierde
last = f'{RUNS}/frutos/weights/last.pt'

if os.path.exists(last):
    print('Retomando entrenamiento desde', last)
    !yolo train resume model={last}
else:
    !yolo task=detect mode=train model=yolo11s.pt data={DATASETS}/DatasetFrutos/data.yaml epochs=150 imgsz=1024 patience=30 batch=-1 plots=True project={RUNS} name=frutos exist_ok=True

## 4. Resultados (validación)

In [ ]:
from IPython.display import Image as IPyImage, display

RUNS = f'{CARPETA_DRIVE}/runs'
for modelo in ('hojas', 'frutos'):
    print('=====', modelo)
    for archivo in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
        ruta = f'{RUNS}/{modelo}/{archivo}'
        if os.path.exists(ruta):
            display(IPyImage(filename=ruta, width=800))
        else:
            print('No existe todavia:', ruta)

## 5. Evaluación final en `test`

Ejecutar **una sola vez**, al final. Para comparar experimentos usa los resultados de validación.

In [ ]:
RUNS = f'{CARPETA_DRIVE}/runs'
!yolo task=detect mode=val split=test model={RUNS}/hojas/weights/best.pt data={DATASETS}/DatasetHojas/data.yaml imgsz=1024 project={RUNS} name=hojas_test exist_ok=True
!yolo task=detect mode=val split=test model={RUNS}/frutos/weights/best.pt data={DATASETS}/DatasetFrutos/data.yaml imgsz=1024 project={RUNS} name=frutos_test exist_ok=True